In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from src.embeddings.embedder import get_model, embed_texts, embed_single, build_movie_text

In [3]:
# Load the cleaned movie dataset from Lab 10
df = pd.read_csv('../data/processed/cleaned/movies_clean.csv')
print(f"Loaded {len(df)} movies")

Loaded 126 movies


In [5]:
print(df[['title', 'overview', 'genre']].head(3))

             title                                           overview   genre
0       Zootopia 2  Officer Judy Hopps and Nick Wilde return to th...  Action
1  Alien: Covenant  The crew of the colony ship Covenant, bound fo...  Sci-Fi
2            David  A sweeping epic retelling of the biblical stor...  Action


In [6]:
# Sample movie descriptions to explore embeddings
sample_texts = [
    "A hero goes on a space adventure to save the galaxy",
    "An astronaut travels through a wormhole in deep space",
    "A detective investigates a series of mysterious murders",
    "Two people fall in love during a summer vacation",
    "A group of soldiers fight in a World War II battle",
]

# Generate embeddings - each text becomes a vector of 384 numbers
embeddings = embed_texts(sample_texts)

print(f"Shape: {embeddings.shape}")  # (5, 384)
print(f"Type:  {type(embeddings)}")
print(f"First 8 numbers of embedding 0: {embeddings[0][:8]}")

Loading model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully
Shape: (5, 384)
Type:  <class 'numpy.ndarray'>
First 8 numbers of embedding 0: [-0.03613125  0.0962556   0.01317967 -0.00041125 -0.05341054  0.0536692
  0.05146044  0.06007211]


In [7]:
from sentence_transformers import util
sim_matrix = util.cos_sim(embeddings, embeddings)

print("Cosine Similarity Matrix:")
print("(Rows and columns match the sample_texts list above)")
print()
labels = ["Space hero", "Astronaut", "Detective", "Romance", "War"]
for i, row_label in enumerate(labels):
    scores = [f"{sim_matrix[i][j].item():.2f}" for j in range(len(labels))]
    print(f"{row_label:12s}: {' | '.join(scores)}")

print()
print("Expected: Space hero vs Astronaut should have the highest score (~0.7+)")

Cosine Similarity Matrix:
(Rows and columns match the sample_texts list above)

Space hero  : 1.00 | 0.41 | 0.03 | 0.11 | 0.11
Astronaut   : 0.41 | 1.00 | 0.11 | 0.03 | 0.02
Detective   : 0.03 | 0.11 | 1.00 | 0.08 | -0.01
Romance     : 0.11 | 0.03 | 0.08 | 1.00 | 0.03
War         : 0.11 | 0.02 | -0.01 | 0.03 | 1.00

Expected: Space hero vs Astronaut should have the highest score (~0.7+)


In [8]:
import torch

query = "exciting science fiction adventure in outer space"
query_embedding = embed_single(query)
scores = util.cos_sim(query_embedding, embeddings)[0]
top_results = torch.topk(scores, k=len(sample_texts))

print(f"Query: '{query}'")
print()
print("Ranked results:")
for rank, (score, idx) in enumerate(zip(top_results.values, top_results.indices)):
    print(f"  {rank+1}. [{score:.4f}] {sample_texts[idx]}")

Query: 'exciting science fiction adventure in outer space'

Ranked results:
  1. [0.5127] A hero goes on a space adventure to save the galaxy
  2. [0.3148] An astronaut travels through a wormhole in deep space
  3. [0.1711] A detective investigates a series of mysterious murders
  4. [0.1669] Two people fall in love during a summer vacation
  5. [0.1195] A group of soldiers fight in a World War II battle


In [9]:
from sklearn.metrics.pairwise import euclidean_distances

text_a = "A hero goes on a space adventure to save the galaxy"
text_b = "An astronaut travels through a wormhole in deep space"
text_c = "Two people fall in love during a summer vacation"

emb_a = embed_single(text_a, normalize=True)
emb_b = embed_single(text_b, normalize=True)
emb_c = embed_single(text_c, normalize=True)

cos_ab = util.cos_sim(emb_a, emb_b).item()
cos_ac = util.cos_sim(emb_a, emb_c).item()

dot_ab = float(np.dot(emb_a, emb_b))
dot_ac = float(np.dot(emb_a, emb_c))

euc_ab = float(np.linalg.norm(emb_a - emb_b))
euc_ac = float(np.linalg.norm(emb_a - emb_c))

print("Comparing 'Space adventure' (A) with:")
print(f"  B = 'Astronaut wormhole': cosine={cos_ab:.4f}, dot={dot_ab:.4f}, euclidean={euc_ab:.4f}")
print(f"  C = 'Summer romance':     cosine={cos_ac:.4f}, dot={dot_ac:.4f}, euclidean={euc_ac:.4f}")
print()
print("Conclusion: A and B should be more similar than A and C across all measures.")

Comparing 'Space adventure' (A) with:
  B = 'Astronaut wormhole': cosine=0.4145, dot=0.4145, euclidean=1.0821
  C = 'Summer romance':     cosine=0.1064, dot=0.1064, euclidean=1.3369

Conclusion: A and B should be more similar than A and C across all measures.


In [10]:
from src.embeddings.chroma_store import get_chroma_client, get_collection, add_movies_to_collection

client = get_chroma_client()
collection = get_collection(client=client, reset=False)

print(f"Collection '{collection.name}' ready")
print(f"Current count: {collection.count()} movies")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Collection 'movies' ready
Current count: 0 movies


In [11]:
df = pd.read_csv('../data/processed/cleaned/movies_clean.csv')

if 'primary_genre' not in df.columns:
    df['primary_genre'] = df['genres'].apply(
        lambda x: str(x).split(',')[0].strip() if pd.notna(x) else 'Unknown'
    )

total = add_movies_to_collection(df, collection, batch_size=100)
print(f"Done. {total} movies are now searchable by meaning.")

Collection already has 0 movies
Collection now contains 126 movies
Done. 126 movies are now searchable by meaning.


In [14]:
results = collection.query(
    query_texts=["exciting adventure in outer space"],
    n_results=3
)

print("Top 3 results for 'exciting adventure in outer space':")
print()
for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
)):
    similarity = 1 - dist 
    print(f"{i+1}. [{similarity:.3f}] {meta['title']} ({meta['year']}) - {meta['genre']}")
    # Print first 120 characters of the document text
    print(f"{doc[:120]}...")
    print()

Top 3 results for 'exciting adventure in outer space':

1. [0.479] Interstellar (2014) - Action
Interstellar | With Earth's agricultural systems collapsing and humanity facing extinction, a former NASA pilot leads a ...

2. [0.454] Project Hail Mary (2026) - Action
Project Hail Mary | A lone astronaut wakes aboard a spacecraft with no memory of his name or mission, only to piece toge...

3. [0.410] The Super Mario Galaxy Movie (2026) - Family
The Super Mario Galaxy Movie | The young Air Nomad Aang, destined to be the Avatar who can master all four elements and ...



In [15]:
queries = [
    "romantic love story",
    "scary horror film with ghosts",
    "animated movie for children",
]

results = collection.query(
    query_texts=queries,
    n_results=3
)

for q_idx, query in enumerate(queries):
    print(f"Query: '{query}'")
    for rank in range(3):
        meta = results["metadatas"][q_idx][rank]
        dist = results["distances"][q_idx][rank]
        sim  = 1 - dist
        print(f"  {rank+1}. [{sim:.3f}] {meta['title']} ({meta['year']})")
    print()

Query: 'romantic love story'
  1. [0.446] Your Heart Will Be Broken (2026)
  2. [0.340] Hemmet (1972)
  3. [0.339] Starbright (2026)

Query: 'scary horror film with ghosts'
  1. [0.543] Scream 7 (2026)
  2. [0.507] Lee Cronin's The Mummy (2026)
  3. [0.476] The House on Haunted Grounds (2026)

Query: 'animated movie for children'
  1. [0.462] The Super Mario Bros. Movie (2023)
  2. [0.423] Outcome (2026)
  3. [0.421] The Super Mario Galaxy Movie (2026)



Filter by Genre

In [16]:
results = collection.query(
    query_texts=["intense fighting and explosions"],
    where={"genre": "Action"},  
    n_results=5
)

print("Action movies about 'intense fighting and explosions':")
for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
    print(f"  [{1-dist:.3f}] {meta['title']} ({meta['year']})")

Action movies about 'intense fighting and explosions':
  [0.409] War of the Worlds (2025)
  [0.352] Turbulence (2025)
  [0.343] Hellfire (2026)
  [0.333] One Battle After Another (2025)
  [0.323] War Machine (2026)


Filter by Year Range

In [17]:
# Find recent movies (2010 and later) about crime
results = collection.query(
    query_texts=["crime thriller with a plot twist"],
    where={"year": {"$gte": 2010}},  
    n_results=5
)

print("Recent crime thrillers (2010+):")
for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
    print(f"  [{1-dist:.3f}] {meta['title']} ({meta['year']}) - rating: {meta['rating']}")

Recent crime thrillers (2010+):
  [0.481] Crime 101 (2026) - rating: 6.5
  [0.405] Bloodline Killer (2024) - rating: 5.45
  [0.372] Outcome (2026) - rating: 5.4
  [0.372] The Unknown Man (2021) - rating: 7.04
  [0.357] Inception (2010) - rating: 8.8


Filter by Multiple Conditions

In [22]:
results = collection.query(
    query_texts=["funny family comedy"],
    where={
        "$and": [
            {"genre": "Comedy"},
            {"rating": {"$gte": 7.0}},
            {"year":   {"$gte": 2010}},
            {"year":   {"$lte": 2026}},
        ]
    },
    n_results=5
)

print("Highly-rated comedies from 2010-2026:")
for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
    print(f" [{1-dist:.3f}] {meta['title']} ({meta['year']}, rating={meta['rating']})")

Highly-rated comedies from 2010-2026:
 [0.171] ¿Quieres Ser Mi Novia? (2026, rating=9.15)


In [24]:
from src.embeddings.chroma_store import get_chroma_client, get_collection
from src.embeddings.search_engine import compare_search

In [25]:
df = pd.read_csv('../data/processed/cleaned/movies_clean.csv')
if 'primary_genre' not in df.columns:
    df['primary_genre'] = df['genres'].apply(
        lambda x: str(x).split(',')[0].strip() if pd.notna(x) else 'Unknown'
    )

client = get_chroma_client()
collection = get_collection(client)
test_queries = [
    "superhero saves the world",
    "a lonely person finds friendship",
    "war and tragedy in Europe",
    "mystery murder investigation",
]

for q in test_queries:
    results = compare_search(q, df, collection=collection, n_results=3)
    print("=" * 60)

--- Query: 'superhero saves the world' ---

Keyword search found 0 results:

Semantic search found 3 results:
  [0.448] Avengers: Doomsday (2026) - Sci-Fi
  [0.435] The Avengers (2012) - Sci-Fi
  [0.411] The Dark Knight (2008) - Action
--- Query: 'a lonely person finds friendship' ---

Keyword search found 0 results:

Semantic search found 3 results:
  [0.385] Send Help (2026) - Action
  [0.380] Roommates (2026) - Comedy
  [0.336] Starbright (2026) - Adventure
--- Query: 'war and tragedy in Europe' ---

Keyword search found 0 results:

Semantic search found 3 results:
  [0.346] War of the Worlds (2025) - Action
  [0.302] 180 (2026) - Thriller
  [0.301] One Battle After Another (2025) - Action
--- Query: 'mystery murder investigation' ---

Keyword search found 0 results:

Semantic search found 3 results:
  [0.572] Bloodline Killer (2024) - Horror
  [0.534] The Unknown Man (2021) - Action
  [0.508] The Mortuary Assistant (2026) - Horror


In [26]:
from src.embeddings.search_engine import keyword_search, semantic_search

In [27]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

test_cases = [
    {"query": "space exploration",     "synonym": "interstellar journey"},
    {"query": "falling in love",        "synonym": "romantic relationship"},
    {"query": "criminal investigation", "synonym": "detective mystery"},
]

for case in test_cases:
    q = case["query"]
    q2 = case["synonym"]

    kw1 = keyword_search(q,  df, n_results=5)
    kw2 = keyword_search(q2, df, n_results=5)
    sem1 = semantic_search(q,  n_results=5, collection=collection)
    sem2 = semantic_search(q2, n_results=5, collection=collection)

    titles_kw1  = set(kw1['title'].tolist())
    titles_kw2  = set(kw2['title'].tolist())
    titles_sem1 = set(sem1['title'].tolist())
    titles_sem2 = set(sem2['title'].tolist())

    kw_overlap  = len(titles_kw1  & titles_kw2)
    sem_overlap = len(titles_sem1 & titles_sem2)

    print(f"Query pair: '{q}' vs '{q2}'")
    print(f"  Keyword overlap:  {kw_overlap}/5 movies in common")
    print(f"  Semantic overlap: {sem_overlap}/5 movies in common")
    print("  (Higher overlap = more consistent results across synonyms)")
    print()


Query pair: 'space exploration' vs 'interstellar journey'
  Keyword overlap:  0/5 movies in common
  Semantic overlap: 3/5 movies in common
  (Higher overlap = more consistent results across synonyms)

Query pair: 'falling in love' vs 'romantic relationship'
  Keyword overlap:  0/5 movies in common
  Semantic overlap: 3/5 movies in common
  (Higher overlap = more consistent results across synonyms)

Query pair: 'criminal investigation' vs 'detective mystery'
  Keyword overlap:  0/5 movies in common
  Semantic overlap: 2/5 movies in common
  (Higher overlap = more consistent results across synonyms)



In [28]:
from src.embeddings.hybrid_search import hybrid_search

In [29]:
query = "action hero saves people"
n = 5

# Individual results
kw  = keyword_search(query, df, n_results=n)
sem = semantic_search(query, collection=collection, n_results=n)
hyb = hybrid_search(query, df, collection, n_results=n)

print(f"KEYWORD SEARCH ({len(kw)} results):")
for _, row in kw.iterrows():
    print(f"  {row['title']} ({row['year']})")

print()
print(f"SEMANTIC SEARCH ({len(sem)} results):")
for _, row in sem.iterrows():
    print(f"  [{row['similarity']:.3f}] {row['title']} ({row['year']})")

print()
print(f"HYBRID SEARCH / RRF ({len(hyb)} results):")
for _, row in hyb.iterrows():
    print(f"  [rrf={row['rrf_score']:.4f}] {row['title']} ({row['year']})")

KEYWORD SEARCH (0 results):

SEMANTIC SEARCH (5 results):
  [0.390] Send Help (2026)
  [0.388] Mercy (2026)
  [0.325] Passenger 57 (1992)
  [0.325] Good Luck, Have Fun, Don't Die (2026)
  [0.302] Play Dead (2025)

HYBRID SEARCH / RRF (5 results):
  [rrf=0.0164] Send Help (2026)
  [rrf=0.0161] Mercy (2026)
  [rrf=0.0159] Passenger 57 (1992)
  [rrf=0.0156] Good Luck, Have Fun, Don't Die (2026)
  [rrf=0.0154] Play Dead (2025)


Part 13 - Analytical Questions and Examples

Question 1: Which movies are most similar to a specific title?

In [30]:
sample_movie = df[df['title'].str.contains('Avatar', na=False)].iloc[0]
movie_text = build_movie_text(sample_movie)

print(f"Finding movies similar to: {sample_movie['title']}")
print(f"Text used: {movie_text[:150]}...")
print()

results = semantic_search(movie_text, n_results=6, collection=collection)

print("Most similar movies:")
for _, row in results.iloc[1:].iterrows():
    print(f"  [{row['similarity']:.3f}] {row['title']} ({row['year']}) - {row['genre']}")

Finding movies similar to: Avatar: Fire and Ash
Text used: Avatar: Fire and Ash | Jake Sully and Ney'tiri face a devastating new threat on Pandora when a volcanic Ash People clan, shaped by the planet's most e...

Most similar movies:
  [0.473] Avatar (2009) - Sci-Fi
  [0.429] The Super Mario Galaxy Movie (2026) - Family
  [0.406] Avatar: Aang, The Last Airbender (2026) - Animation
  [0.330] The Avengers (2012) - Sci-Fi
  [0.318] Titanic (1997) - Romance


Question 2: Which genres have the most semantically diverse movies?

In [31]:
from src.embeddings.embedder import embed_texts, build_movie_text

sample = df.sample(n=min(500, len(df)), random_state=42).copy()
sample['text'] = sample.apply(build_movie_text, axis=1)

print("Generating embeddings for sample movies...")
embs = embed_texts(sample['text'].tolist())
sample['embedding'] = list(embs)

if 'primary_genre' not in sample.columns:
    sample['primary_genre'] = sample['genres'].apply(
        lambda x: str(x).split(',')[0].strip() if pd.notna(x) else 'Unknown'
    )

genre_diversity = {}
for genre, group in sample.groupby('primary_genre'):
    if len(group) < 5:
        continue
    emb_matrix = np.vstack(group['embedding'].values)
    # Average variance across all embedding dimensions
    diversity = float(np.mean(np.var(emb_matrix, axis=0)))
    genre_diversity[genre] = round(diversity, 6)

top_genres = sorted(genre_diversity.items(), key=lambda x: -x[1])[:10]
print("Most semantically diverse genres:")
for genre, score in top_genres:
    print(f"  {genre:20s}: diversity = {score:.6f}")

Generating embeddings for sample movies...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Most semantically diverse genres:
  Action              : diversity = 0.002195
  Horror              : diversity = 0.001945
  Sci-Fi              : diversity = 0.001822
  Comedy              : diversity = 0.001796


Question 3: How do search results change with filters?

In [32]:
query = "adventure and discovery"
unfiltered = semantic_search(query, n_results=5, collection=collection)
action_only = semantic_search(query, n_results=5, collection=collection, filters={"genre": "Adventure"})
recent_only = semantic_search(query, n_results=5, collection=collection, filters={"year": {"$gte": 2010}})

print("No filter:")
print(unfiltered[['title', 'year', 'genre', 'similarity']].to_string(index=False))

print()
print("Adventure genre only:")
print(action_only[['title', 'year', 'genre', 'similarity']].to_string(index=False))

print()
print("2010 and later only:")
print(recent_only[['title', 'year', 'genre', 'similarity']].to_string(index=False))

No filter:
              title  year  genre  similarity
            Outcome  2026 Comedy      0.3628
            Shelter  2026 Action      0.3365
         Zootopia 2  2025 Action      0.3325
"Wuthering Heights"  2026 Action      0.3273
  War of the Worlds  2025 Action      0.3247

Adventure genre only:
                 title  year     genre  similarity
            Starbright  2026 Adventure      0.1181
Avengers: Infinity War  2018 Adventure      0.1103

2010 and later only:
              title  year  genre  similarity
            Outcome  2026 Comedy      0.3628
            Shelter  2026 Action      0.3365
         Zootopia 2  2025 Action      0.3325
"Wuthering Heights"  2026 Action      0.3273
  War of the Worlds  2025 Action      0.3247
